In [ ]:
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

load_dotenv()

RAW_AQ   = "../data/raw/openaq/"
RAW_TR   = "../data/raw/trends/"
INTERIM  = "../data/interim/"
PROCESSED = "../data/processed/"

CITIES = ["bangkok", "krakow", "melbourne", "lahore"]
TERMS  = ["air_quality", "air_purifier", "N95_mask"]
HEALTH_TERMS = ["hospital", "breathing_difficulty", "cough_medicine", "eye_drops", "chest_pain"]
ALL_TERMS = TERMS + HEALTH_TERMS

print("Ready ✓")

In [ ]:
def clean_aq(city):
    df = pd.read_csv(f"{RAW_AQ}{city}_raw.csv")
    df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None)

    df = df[(df["pm25"] >= 0) & (df["pm25"] <= 500)]

    df = df[df["percent_complete"] >= 50]

    df["week"] = df["date"].dt.to_period("W-SAT").dt.start_time
    weekly = (
        df.groupby("week")["pm25"]
        .mean()
        .reset_index()
    )
    weekly.columns = ["week", "pm25_mean"]
    weekly["city"] = city

    print(f"{city}: {len(weekly)} weeks | "
          f"PM2.5 {weekly['pm25_mean'].min():.1f}–{weekly['pm25_mean'].max():.1f} µg/m³")
    return weekly

aq_all = pd.concat([clean_aq(c) for c in CITIES], ignore_index=True)

aq_all.to_csv(f"{INTERIM}aq_weekly_all.csv", index=False)
print(f"\n✓ Saved interim AQ weekly → {len(aq_all)} total rows")

In [ ]:
def load_trends(city, term):
    df = pd.read_csv(f"{RAW_TR}trends_{city}_{term}.csv")
    df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None)
    df = df.rename(columns={"date": "week", "search_index": term})
    df["week"] = df["week"].dt.to_period("W-SAT").dt.start_time
    return df.set_index("week")

def load_all_trends(city):
    available = [t for t in ALL_TERMS if os.path.exists(f"{RAW_TR}trends_{city}_{t}.csv")]
    dfs = [load_trends(city, t) for t in available]
    combined = pd.concat(dfs, axis=1).reset_index()
    for t in ALL_TERMS:
        if t not in available:
            combined[t] = 0
            print(f"{city}: no Trends data for {t} — filled with 0")
    combined = combined[["week"] + ALL_TERMS]
    combined["city"] = city
    print(f"{city}: {len(combined)} weeks, "
          f"columns: {[c for c in combined.columns if c not in ['week','city']]}")
    return combined

trends_all = pd.concat([load_all_trends(c) for c in CITIES], ignore_index=True)
trends_all.to_csv(f"{INTERIM}trends_weekly_all.csv", index=False)
print(f"\n✓ Saved interim Trends weekly → {len(trends_all)} total rows")

In [ ]:
merged = pd.merge(aq_all, trends_all, on=["week", "city"], how="inner")
merged = merged.sort_values(["city", "week"]).reset_index(drop=True)

merged.to_csv(f"{PROCESSED}all_cities_combined.csv", index=False)

print(f"✓ Saved → all_cities_combined.csv")
print(f"\nShape: {merged.shape}")
print(f"\nColumns: {list(merged.columns)}")
print(f"\nDate range: {merged['week'].min()} → {merged['week'].max()}")
print(f"\nRows per city:")
print(merged.groupby("city").size())
print(f"\nSample:")
print(merged.head(3))

In [ ]:
fig, axes = plt.subplots(len(CITIES), 1, figsize=(14, 4 * len(CITIES)))

city_colors = {
    "bangkok":   {"pm25": "#d62728", "search": "#1f77b4"},
    "krakow":    {"pm25": "#d62728", "search": "#2ca02c"},
    "melbourne": {"pm25": "#d62728", "search": "#9467bd"},
    "lahore":    {"pm25": "#d62728", "search": "#ff7f0e"}
}

for ax, city in zip(axes, CITIES):
    df = merged[merged["city"] == city].copy()
    df["week"] = pd.to_datetime(df["week"])

    ax2 = ax.twinx()

    ax.plot(df["week"], df["pm25_mean"],
            color=city_colors[city]["pm25"],
            linewidth=1.5, label="PM2.5 (µg/m³)")
    ax.fill_between(df["week"], df["pm25_mean"],
                    alpha=0.15, color=city_colors[city]["pm25"])

    ax2.plot(df["week"], df["air_quality"],
             color=city_colors[city]["search"],
             linewidth=1.5, linestyle="--", label="Search: 'air quality'")

    ax.set_ylabel("PM2.5 (µg/m³)", color=city_colors[city]["pm25"])
    ax2.set_ylabel("Search Index (0–100)", color=city_colors[city]["search"])
    ax.set_title(f"{city.capitalize()} — PM2.5 vs 'air quality' search interest",
                 fontsize=12, fontweight="bold")

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)

    ax.tick_params(axis="y", labelcolor=city_colors[city]["pm25"])
    ax2.tick_params(axis="y", labelcolor=city_colors[city]["search"])
    ax.set_xlabel("")

plt.tight_layout(pad=3.0)
plt.savefig("../outputs/figures/timeseries_pm25_vs_search.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/timeseries_pm25_vs_search.png")

In [ ]:
from scipy import stats

print("=== PEARSON CORRELATION: PM2.5 vs Search Terms ===\n")

for city in CITIES:
    df = merged[merged["city"] == city].dropna()
    print(f"{city.upper()}")
    for term in ALL_TERMS:
        r, p = stats.pearsonr(df["pm25_mean"], df[term])
        sig = "✓ significant" if p < 0.05 else "✗ not significant"
        print(f"  {term:20s} → r={r:+.3f}, p={p:.4f} {sig}")
    print()

In [ ]:
from scipy.signal import correlate

print("=== CROSS-CORRELATION WITH LAGS (weeks) ===\n")
MAX_LAG = 8

for city in CITIES:
    df = merged[merged["city"] == city].dropna().copy()
    print(f"{city.upper()}")

    for term in ALL_TERMS:
        series = df[term]
        if series.std() == 0:
            print(f"  {term:20s} → SKIPPED (constant series)")
            continue

        pm25_norm = (df["pm25_mean"] - df["pm25_mean"].mean()) / df["pm25_mean"].std()
        srch_norm = (series - series.mean()) / series.std()

        cc = correlate(srch_norm, pm25_norm, mode="full")
        lags = np.arange(-(len(pm25_norm)-1), len(pm25_norm))

        mask = (lags >= -MAX_LAG) & (lags <= MAX_LAG)
        cc_trim = cc[mask]
        lags_trim = lags[mask]

        peak_lag = lags_trim[np.argmax(cc_trim)]
        peak_corr = cc_trim.max() / len(pm25_norm)

        direction = "search LEADS pollution" if peak_lag < 0 else \
                    "search LAGS pollution" if peak_lag > 0 else \
                    "simultaneous"

        print(f"  {term:20s} → peak lag={peak_lag:+d} weeks "
              f"(r={peak_corr:.3f}) — {direction}")
    print()

In [ ]:
fig, axes = plt.subplots(1, len(CITIES), figsize=(5.3 * len(CITIES), 5))

for ax, city in zip(axes, CITIES):
    df = merged[merged["city"] == city].dropna().copy()
    pm25_norm = (df["pm25_mean"] - df["pm25_mean"].mean()) / df["pm25_mean"].std()

    for term, color in zip(TERMS, ["#1f77b4", "#ff7f0e", "#2ca02c"]):
        series = df[term]
        if series.std() == 0:
            continue
        srch_norm = (series - series.mean()) / series.std()
        cc = correlate(srch_norm, pm25_norm, mode="full")
        lags = np.arange(-(len(pm25_norm)-1), len(pm25_norm))
        mask = (lags >= -8) & (lags <= 8)
        cc_norm = cc[mask] / len(pm25_norm)
        ax.plot(lags[mask], cc_norm, marker="o", markersize=4,
                label=term, color=color)

    ax.axvline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.axhline(0, color="black", linestyle="-", linewidth=0.5, alpha=0.3)
    ax.set_title(f"{city.capitalize()}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Lag (weeks)\n← search leads | pollution leads →")
    ax.set_ylabel("Cross-correlation")
    ax.legend(fontsize=8)
    ax.set_xticks(range(-8, 9))
    ax.grid(alpha=0.3)

plt.suptitle("Cross-correlation: PM2.5 vs Search Terms by Lag",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../outputs/figures/crosscorr_lag_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/crosscorr_lag_plot.png")

In [ ]:
from statsmodels.tsa.seasonal import STL

fig, axes = plt.subplots(len(CITIES), 2, figsize=(16, 4.6 * len(CITIES)))

for row, city in enumerate(CITIES):
    df = merged[merged["city"] == city].copy()
    df["week"] = pd.to_datetime(df["week"])
    df = df.set_index("week").asfreq("W-SUN")

    for col, (series_col, label) in enumerate([
        ("pm25_mean", "PM2.5 (µg/m³)"),
        ("air_quality", "Search: 'air quality'")
    ]):
        ax = axes[row][col]
        series = df[series_col].interpolate()

        if len(series) < 2 * 52:
            ax.text(0.5, 0.5, "Insufficient data for STL (needs 104+ weeks)",
                    ha="center", va="center", transform=ax.transAxes, color="grey")
            ax.set_title(f"{city.capitalize()} — {label}", fontweight="bold")
            ax.grid(alpha=0.3)
            continue

        stl = STL(series, period=52, robust=True)
        result = stl.fit()

        ax.plot(df.index, series, color="lightgrey", linewidth=1, label="Original")
        ax.plot(df.index, result.trend, color="#d62728", linewidth=2, label="Trend")
        ax.fill_between(df.index,
                        result.trend - result.seasonal,
                        result.trend + result.seasonal,
                        alpha=0.2, color="#1f77b4", label="Seasonal band")

        ax.set_title(f"{city.capitalize()} — {label}", fontweight="bold")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

plt.suptitle("STL Decomposition: Trend + Seasonal Component",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../outputs/figures/stl_decomposition.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/stl_decomposition.png")

In [ ]:
YEARS = [2022, 2023, 2024]

fig, axes = plt.subplots(len(CITIES), 1, figsize=(14, 2.6 * len(CITIES)))

for ax, city in zip(axes, CITIES):
    df = aq_all[aq_all["city"] == city].copy()
    df["week"] = pd.to_datetime(df["week"])
    df["year"] = df["week"].dt.year
    df["woy"] = ((df["week"].dt.dayofyear - 1) // 7 + 1).clip(upper=52)
    df = df[df["year"].isin(YEARS)]

    grid = (
        df.pivot_table(index="year", columns="woy", values="pm25_mean", aggfunc="mean")
        .reindex(index=YEARS, columns=range(1, 53))
    )

    ax.set_facecolor("#e6e6e6")
    im = ax.imshow(grid.values.astype(float), aspect="auto", cmap="Reds",
                   interpolation="nearest")

    ax.set_xticks(range(0, 52, 4))
    ax.set_xticklabels(range(1, 53, 4))
    ax.set_yticks(range(len(YEARS)))
    ax.set_yticklabels(YEARS)
    ax.set_ylabel("Year")
    ax.set_title(f"{city.capitalize()} — weekly mean PM2.5", fontsize=12, fontweight="bold")
    fig.colorbar(im, ax=ax, label="PM2.5 (µg/m³)", pad=0.01)

axes[-1].set_xlabel("Week of year")
plt.suptitle("PM2.5 Seasonality: Week of Year × Year",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../outputs/figures/seasonality_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/seasonality_heatmap.png")